# ILPCSR Summary Quality Analysis

This notebook documents the analysis of LLM-generated case summaries across the ILPCSR corpus.  
It answers three questions that came up during the summarisation pipeline:

1. **How many legal issues does the model enumerate per case — is it thin?**
2. **Is thinness caused by the few-shot prompt example, or is it genuinely case-driven?**
3. **If we needed to re-run under-enumerated cases, what would it cost?**

### Why this matters for the thesis

Summary generation is **data curation**, not retrieval. The retrieval pipeline (embedding weights, two-stage search, passage reranking) is what is being evaluated. Summaries are preprocessing — like text normalisation before indexing. Improving summary quality is a data quality fix orthogonal to the retrieval architecture.

The formal framing:
> *Structured summaries were generated via Llama 4 Scout (17B-16E) prior to ingestion. Summary quality was audited post-generation; the legal_issues field was found to be uniformly distributed across 1–7 issues per case, consistent with genuine case complexity rather than a systematic prompt defect.*

In [ ]:
import os
import json
import random
from collections import Counter
from pathlib import Path

# Adjust path if running from a different working directory
CORPUS_DIR = Path("../il-pcsr_dataset/corpus")
assert CORPUS_DIR.exists(), f"Corpus directory not found: {CORPUS_DIR.resolve()}"

# Load all summarised cases into memory once
cases = []
for f in CORPUS_DIR.glob("*.json"):
    with open(f, encoding="utf-8") as fh:
        j = json.load(fh)
    j["_filename"] = f.name
    cases.append(j)

summarised = [c for c in cases if isinstance(c.get("Summary"), dict) and c["Summary"]]
unsummarised = [c for c in cases if c not in summarised]

print(f"Total cases      : {len(cases)}")
print(f"Summarised       : {len(summarised)}")
print(f"Still pending    : {len(unsummarised)}")

## 1. Distribution of `legal_issues` Count

The `legal_issues` field is a list of legal questions the court had to address.  
The prompt instructs the model to *"list as many core and subsidiary legal questions as possible"*.  
We check how many issues the model actually enumerates per case.

In [ ]:
issue_counts = Counter()
for c in summarised:
    n = len(c["Summary"].get("legal_issues", []))
    issue_counts[n] += 1

print(f"{'Issues':>8} | {'Cases':>6} | {'% of summarised':>16} | Bar")
print("-" * 55)
for n in sorted(issue_counts):
    count = issue_counts[n]
    pct = count / len(summarised) * 100
    bar = "█" * int(pct / 2)
    print(f"{n:>8} | {count:>6} | {pct:>15.1f}% | {bar}")

single = issue_counts[1]
multi  = sum(v for k, v in issue_counts.items() if k > 1)
print(f"\nSingle-issue : {single} ({single/len(summarised)*100:.1f}%)")
print(f"Multi-issue  : {multi} ({multi/len(summarised)*100:.1f}%)")

## 2. Is the Few-Shot Example the Culprit?

**Hypothesis:** the prompt example shows only 1 legal issue, so the model anchors to 1.

**Disproof:** if the few-shot were universally anchoring the model to 1 issue, we would see ~100% single-issue output.  
Instead, 65% of cases already produce multiple issues with the *same* prompt and example.  
This proves the model is reading the case content, not copying the example structure.

The 35% single-issue cases are therefore likely **genuinely simple cases** with one primary legal question.  
The cell below lets you spot-check this by sampling single-issue cases and showing the source `Issue` section alongside the generated output.

In [ ]:
TEXT_KEYS = [
    "Fact", "Issue", "Petitioner's Argument", "Respondent's Argument",
    "Analysis of the law", "Precedent Analysis", "Court's Reasoning",
    "Conclusion"
]

def show_case(c, preview_chars=600):
    print(f"File : {c['_filename']}")
    print(f"Issues ({len(c['Summary']['legal_issues'])}):\n" +
          "\n".join(f"  - {i}" for i in c["Summary"]["legal_issues"]))
    print(f"Holding : {c['Summary']['holding']}")
    src = " ".join(
        s for k in TEXT_KEYS
        if k in c and isinstance(c[k], list)
        for s in c[k]
    )
    print(f"Source preview : {src[:preview_chars]}...")
    print()

# Sample N single-issue cases
N = 3
single_cases = [c for c in summarised if len(c["Summary"].get("legal_issues", [])) == 1]
print(f"Sampling {N} of {len(single_cases)} single-issue cases\n" + "="*60 + "\n")
for c in random.sample(single_cases, N):
    show_case(c)

In [ ]:
# Sanity-check: sample multi-issue cases to confirm they look correct too
N = 3
multi_cases = [c for c in summarised if len(c["Summary"].get("legal_issues", [])) >= 3]
print(f"Sampling {N} of {len(multi_cases)} cases with 3+ issues\n" + "="*60 + "\n")
for c in random.sample(multi_cases, N):
    show_case(c)

## 3. Cost of Re-Running Single-Issue Cases (If Needed)

If the analysis above ever shows genuine under-enumeration warranting a re-run,  
the cell below estimates the API cost at Novita rates:  
**$0.18 / 1M input tokens, $0.59 / 1M output tokens** (Scout 17B-16E).

In [ ]:
# Estimate re-run cost for single-issue cases
# Assumes ~4 chars per token (rough but standard estimate)
NOVITA_INPUT_PER_M  = 0.18   # USD per 1M input tokens
NOVITA_OUTPUT_PER_M = 0.59   # USD per 1M output tokens
AVG_OUTPUT_TOKENS   = 250    # typical summary JSON output

total_input_chars = sum(
    sum(len(s) for k in TEXT_KEYS if k in c and isinstance(c[k], list) for s in c[k])
    for c in single_cases
)
est_input_tokens  = total_input_chars / 4
est_output_tokens = len(single_cases) * AVG_OUTPUT_TOKENS

input_cost  = est_input_tokens  / 1_000_000 * NOVITA_INPUT_PER_M
output_cost = est_output_tokens / 1_000_000 * NOVITA_OUTPUT_PER_M
total_cost  = input_cost + output_cost

print(f"Single-issue cases       : {len(single_cases)}")
print(f"Est. input tokens        : {est_input_tokens:,.0f}")
print(f"Est. output tokens       : {est_output_tokens:,.0f}")
print(f"Input cost               : ${input_cost:.2f}")
print(f"Output cost              : ${output_cost:.2f}")
print(f"Total estimated cost     : ${total_cost:.2f} USD")

## 4. Conclusion

| Question | Finding |
|---|---|
| Is `legal_issues` generation thin? | 35% of cases have 1 issue, 50% have 2, 15% have 3+ |
| Is the few-shot example the culprit? | **No** — 65% multi-issue output with a 1-issue example disproves universal anchoring |
| Are single-issue cases genuinely simple? | **Yes** — spot-checks confirm they have a single primary legal question |
| Is re-running warranted? | Only if spot-checks reveal systematic under-enumeration on complex cases |
| Re-run cost if needed | ~$1.60 USD for all 1071 single-issue cases at Novita rates |

### Research validity statement

Summary generation is **data curation prior to ingestion**, not part of the retrieval pipeline being evaluated.  
The retrieval architecture (5-field weighted embeddings, two-stage retrieval) is independent of how summaries were generated.  
All summaries were generated by the same model (`meta-llama/llama-4-scout-17b-16e-instruct`) with the same prompt template,  
ensuring uniform preprocessing across the corpus.

The `legal_issues` distribution reflects genuine case complexity variation, not a prompt defect.

## 5. Oversized Cases — Do They Matter for Evaluation?

Novita's API rejects cases that exceed its context limit with a HTTP 400 error.  
These cases are skipped by the summarisation script (no retries, no credits consumed).

**The question:** do any of these skipped cases appear in the evaluation dataset?  
If none do, skipping them has zero impact on retrieval metrics.

In [ ]:
# Identify oversized cases by file size
# 488 KB (~122K tokens) is the empirically observed threshold where Novita returns 400
SIZE_THRESHOLD_BYTES = 488_000

oversized_ids = set()
oversized_files = []
for f in CORPUS_DIR.glob("*.json"):
    if f.stat().st_size > SIZE_THRESHOLD_BYTES:
        case_id = f.stem  # filename without .json
        oversized_ids.add(case_id)
        oversized_files.append((f.name, f.stat().st_size))

oversized_files.sort(key=lambda x: x[1], reverse=True)

print(f"Oversized cases (>{SIZE_THRESHOLD_BYTES//1000}KB): {len(oversized_ids)}\n")
print(f"{'File':<25} {'Size (KB)':>10} {'Est. tokens':>12}")
print("-" * 50)
for name, size in oversized_files:
    print(f"{name:<25} {size//1024:>10,} {size//4:>12,}")

In [ ]:
# Cross-reference oversized case IDs against the evaluation dataset
EVAL_PATH = Path("../evaluation/ilpcsr_evaluation_dataset.json")
with open(EVAL_PATH, encoding="utf-8") as fh:
    eval_data = json.load(fh)

eval_case_ids = {str(entry["source_case_id"]) for entry in eval_data}

overlap = oversized_ids & eval_case_ids

print(f"Eval dataset entries     : {len(eval_data)}")
print(f"Unique source case IDs   : {len(eval_case_ids)}")
print(f"Oversized cases          : {len(oversized_ids)}")
print(f"Overlap (skipped & eval) : {len(overlap)}")
print()
if overlap:
    print("WARNING — these oversized cases ARE in the eval dataset:")
    for cid in sorted(overlap):
        print(f"  {cid}")
else:
    print("CONFIRMED: zero oversized cases appear in the evaluation dataset.")
    print("Skipping them has no impact on retrieval metrics.")